In [3]:
%pip install matplotlib
%pip install seaborn
%pip install mysql-connector-python
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import mysql.connector

from sqlalchemy import create_engine

In [5]:
tracks = pd.read_csv('../data/tracks.csv')

artists = pd.read_csv('../data/artists.csv')

In [6]:
print(tracks.shape)
print(artists.shape)

(586672, 20)
(1162095, 5)


In [7]:
tracks.head()

,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,35iwgR4jXetI318WEWsa1Q,Carve,6,126903,0,['Uli'],['45tIt06XoI0Iio4LBEVpls'],1922-02-22,0.645,0.4450,0,-13.338,1,0.4510,0.674,0.7440,0.151,0.127,104.851,3
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,0,98200,0,['Fernando Pessoa'],['14jtPCOoNZwquk5wd9DxrY'],1922-06-01,0.695,0.2630,0,-22.136,1,0.9570,0.797,0.0000,0.148,0.655,102.009,1
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,0,181640,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.434,0.1770,1,-21.180,1,0.0512,0.994,0.0218,0.212,0.457,130.418,5
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,0,176907,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.321,0.0946,7,-27.961,1,0.0504,0.995,0.9180,0.104,0.397,169.980,3
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,0,163080,0,['Dick Haymes'],['3BiJGZsyX9sJchTqcSA7Su'],1922,0.402,0.1580,3,-16.900,0,0.0390,0.989,0.1300,0.311,0.196,103.220,4


In [8]:
artists.head()

,id,followers,genres,name,popularity
0,0DheY5irMjBUeLybbCUEZ2,0.0,[],Armid & Amir Zare Pashai feat. Sara Rouzbehani,0
1,0DlhY15l3wsrnlfGio2bjU,5.0,[],ปูนา ภาวิณี,0
2,0DmRESX2JknGPQyO15yxg7,0.0,[],Sadaa,0
3,0DmhnbHjm1qw6NCYPeZNgJ,0.0,[],Tra'gruda,0
4,0Dn11fWM7vHQ3rinvWEl4E,2.0,[],Ioannis Panoutsopoulos,0


In [9]:
tracks = tracks.sample(
    50000,
    random_state=42
)

In [10]:
print(tracks.shape)

(50000, 20)


In [11]:
tracks = tracks.drop_duplicates()

artists = artists.drop_duplicates()

In [12]:
tracks['release_date'] = pd.to_datetime(
    tracks['release_date'],
    format='mixed',
    errors='coerce'
)

In [13]:
tracks['release_year'] = (
    tracks['release_date']
    .dt.year
)

In [14]:
tracks[['release_date', 'release_year']].head()

,release_date,release_year
241517,1985-01-01,1985
444213,2017-08-25,2017
106480,1992-01-01,1992
141137,1969-01-01,1969
586550,2015-12-25,2015


In [15]:
tracks.isnull().sum()

id                  0
name                4
popularity          0
duration_ms         0
explicit            0
artists             0
id_artists          0
release_date        0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
release_year        0
dtype: int64

In [16]:
tracks = tracks.dropna(
    subset=['release_year']
)

In [17]:
tracks_db = tracks.head(5000).copy()

artists_db = artists.head(5000).copy()

In [18]:
print(tracks_db.shape)
print(artists_db.shape)

(5000, 21)
(5000, 5)


In [19]:
artists_clean = artists_db[[
    'id',
    'name'
]].copy()

In [21]:
artists_clean.columns = [
    'artist_id',
    'artist_name'
]

In [22]:
artists_clean = (
    artists_clean
    .drop_duplicates()
    .dropna()
)

In [23]:
artists_clean.head()

,artist_id,artist_name
0,0DheY5irMjBUeLybbCUEZ2,Armid & Amir Zare Pashai feat. Sara Rouzbehani
1,0DlhY15l3wsrnlfGio2bjU,ปูนา ภาวิณี
2,0DmRESX2JknGPQyO15yxg7,Sadaa
3,0DmhnbHjm1qw6NCYPeZNgJ,Tra'gruda
4,0Dn11fWM7vHQ3rinvWEl4E,Ioannis Panoutsopoulos


In [ ]:
tracks_db['artist_id'] = (
    tracks_db['id_artists']
    .astype(str)
    .str.split(',')
    .str[0]
    .str.replace(r"[\"\'\[\] ]", "", regex=True)
)

In [25]:
tracks_clean = tracks_db[[
    'id',
    'name',
    'artist_id',
    'release_year',
    'popularity',
    'danceability',
    'energy',
    'valence',
    'acousticness',
    'duration_ms',
    'tempo'
]].copy()

In [26]:
tracks_clean.columns = [
    'track_id',
    'track_name',
    'artist_id',
    'release_year',
    'popularity',
    'danceability',
    'energy',
    'valence',
    'acousticness',
    'duration_ms',
    'tempo'
]

In [27]:
tracks_clean = (
    tracks_clean
    .drop_duplicates()
    .dropna()
)

In [28]:
tracks_clean.head()

,track_id,track_name,artist_id,release_year,popularity,danceability,energy,valence,acousticness,duration_ms,tempo
241517,0rTUAMd7huYjda84aceuzl,Дети проходных дворов,2jkl2xJVm71azWAgZKyf42,1985,21,0.890,0.6680,0.942,0.570,105015,137.934
444213,4bMy8NjgkVHize43OSrChu,95,7jLSEPYCYQ5ssWU3BICqrW,2017,37,0.663,0.5510,0.339,0.697,234800,128.992
106480,4X1xn2bjxlaLbx6o3jSEXj,As almal ver is,4utxbudXUuseiCfVJ0B2xM,1992,10,0.376,0.0979,0.236,0.485,147733,90.290
141137,3ZwzDf7V1xlAS9WWuviqIm,"Il tuo mondo (Nono, dobri moj nono)",2r4iOKmxTQU9s361Yt3gq1,1969,29,0.565,0.3270,0.556,0.781,194373,103.256
586550,712Hktc9DnU5u9uOSn1q1Y,Bazaar - Official Sunburn Goa 2015 Anthem,"2wX6xSig4Rig5kZU6ePlWe', '6S3KljEiIOWoLMUyZrkQUc",2015,55,0.582,0.9790,0.196,0.125,172500,128.010


In [29]:
from sqlalchemy import create_engine

DATABASE_URL = (
    "mysql+pymysql://"
    "DocBxbiNM1cbnzj.root:slWnqaS2ilBHBfHl"
    "@gateway01.us-west-2.prod.aws.tidbcloud.com:4000/"
    "spotify_db"
    "?ssl_verify_cert=false&ssl_verify_identity=false"
)

engine = create_engine(
    DATABASE_URL
)

In [30]:
engine = create_engine(
    DATABASE_URL
)

In [31]:
print('Conexión creada correctamente')

Conexión creada correctamente


In [32]:
with engine.connect() as conn:
    print("Conexión OK")

Conexión OK


In [43]:
# Leer los IDs de los artistas que ya existen en la base de datos
with engine.connect() as conn:
    existing_artists = pd.read_sql("SELECT artist_id FROM artists", conn)
    existing_artist_ids = existing_artists['artist_id'].tolist()

# Filtrar el DataFrame para obtener solo los nuevos artistas
new_artists = artists_clean[~artists_clean['artist_id'].isin(existing_artist_ids)]

# Insertar solo los nuevos artistas
if not new_artists.empty:
    new_artists.to_sql(
        name='artists',
        con=engine,
        if_exists='append',
        index=False,
        chunksize=100,
        method='multi'
    )

In [45]:
print('Artists insertados correctamente')

Artists insertados correctamente


In [46]:
# Leer los IDs de las canciones que ya existen en la base de datos
with engine.connect() as conn:
    existing_tracks = pd.read_sql("SELECT track_id FROM tracks", conn)
    existing_track_ids = existing_tracks['track_id'].tolist()

# Filtrar el DataFrame para obtener solo las nuevas canciones
new_tracks = tracks_clean[~tracks_clean['track_id'].isin(existing_track_ids)]

# Insertar solo las nuevas canciones
if not new_tracks.empty:
    new_tracks.to_sql(
        name='tracks',
        con=engine,
        if_exists='append',
        index=False,
        chunksize=100,
        method='multi'
    )

DatabaseError: Execution failed on sql 'INSERT INTO tracks (track_id, track_name, artist_id, release_year, popularity, danceability, energy, valence, acousticness, duration_ms, tempo) VALUES (:track_id_m0, :track_name_m0, :artist_id_m0, :release_year_m0, :popularity_m0, :danceability_m0, :energy_m0, :valence_m0, :acousticness_m0, :duration_ms_m0, :tempo_m0), (:track_id_m1, :track_name_m1, :artist_id_m1, :release_year_m1, :popularity_m1, :danceability_m1, :energy_m1, :valence_m1, :acousticness_m1, :duration_ms_m1, :tempo_m1), (:track_id_m2, :track_name_m2, :artist_id_m2, :release_year_m2, :popularity_m2, :danceability_m2, :energy_m2, :valence_m2, :acousticness_m2, :duration_ms_m2, :tempo_m2), (:track_id_m3, :track_name_m3, :artist_id_m3, :release_year_m3, :popularity_m3, :danceability_m3, :energy_m3, :valence_m3, :acousticness_m3, :duration_ms_m3, :tempo_m3), (:track_id_m4, :track_name_m4, :artist_id_m4, :release_year_m4, :popularity_m4, :danceability_m4, :energy_m4, :valence_m4, :acous